In [1]:
import hashlib
import random
import itertools


def read_doc(file):
    with open(file, "r", encoding="utf8") as f:
        return f.read().lower()

docs = {
    "D1": read_doc("D1.txt"),
    "D2": read_doc("D2.txt"),
    "D3": read_doc("D3.txt"),
    "D4": read_doc("D4.txt")
}

In [2]:
def char_kgrams(text, k):
    return set([text[i:i+k] for i in range(len(text)-k+1)])

def word_kgrams(text, k):
    words = text.split()
    return set([" ".join(words[i:i+k]) for i in range(len(words)-k+1)])


char2 = {d: char_kgrams(t,2) for d,t in docs.items()}
char3 = {d: char_kgrams(t,3) for d,t in docs.items()}
word2 = {d: word_kgrams(t,2) for d,t in docs.items()}


def jaccard(a,b):
    return len(a & b) / len(a | b)

pairs = list(itertools.combinations(docs.keys(),2))

print("\nJACCARD SIMILARITIES\n")

print("\nCHAR 2-GRAM")
for p in pairs:
    print(p, jaccard(char2[p[0]], char2[p[1]]))

print("\nCHAR 3-GRAM")
for p in pairs:
    print(p, jaccard(char3[p[0]], char3[p[1]]))

print("\nWORD 2-GRAM")
for p in pairs:
    print(p, jaccard(word2[p[0]], word2[p[1]]))


JACCARD SIMILARITIES


CHAR 2-GRAM
('D1', 'D2') 0.9811320754716981
('D1', 'D3') 0.8156996587030717
('D1', 'D4') 0.6444444444444445
('D2', 'D3') 0.8
('D2', 'D4') 0.6412698412698413
('D3', 'D4') 0.6529968454258676

CHAR 3-GRAM
('D1', 'D2') 0.977979274611399
('D1', 'D3') 0.5803571428571429
('D1', 'D4') 0.3050847457627119
('D2', 'D3') 0.5680473372781065
('D2', 'D4') 0.30590339892665475
('D3', 'D4') 0.31212381771281167

WORD 2-GRAM
('D1', 'D2') 0.9407665505226481
('D1', 'D3') 0.18234165067178504
('D1', 'D4') 0.03024193548387097
('D2', 'D3') 0.1736641221374046
('D2', 'D4') 0.030303030303030304
('D3', 'D4') 0.01607142857142857


In [3]:
def minhash_signature(shingles, num_hash):

    max_shingle = 10000
    coeffA = random.sample(range(1,max_shingle), num_hash)
    coeffB = random.sample(range(0,max_shingle), num_hash)

    signature = []

    for i in range(num_hash):
        min_hash = float("inf")

        for sh in shingles:
            h = int(hashlib.md5(sh.encode()).hexdigest(),16)
            val = (coeffA[i]*h + coeffB[i]) % max_shingle

            if val < min_hash:
                min_hash = val

        signature.append(min_hash)

    return signature

def approx_jaccard(sig1,sig2):
    count = 0
    for a,b in zip(sig1,sig2):
        if a==b:
            count+=1
    return count/len(sig1)

print("\nMINHASH APPROX JACCARD (3-grams D1 vs D2)")

for t in [20,60,150,300,600]:

    sig1 = minhash_signature(char3["D1"], t)
    sig2 = minhash_signature(char3["D2"], t)

    print("t =",t," similarity =",approx_jaccard(sig1,sig2))


MINHASH APPROX JACCARD (3-grams D1 vs D2)
t = 20  similarity = 0.05
t = 60  similarity = 0.016666666666666666
t = 150  similarity = 0.013333333333333334
t = 300  similarity = 0.04666666666666667
t = 600  similarity = 0.02666666666666667


In [4]:
def lsh_prob(s,r,b):
    return 1 - (1 - s**b)**r

print("\nLSH PROBABILITIES")

r = 8
b = 20

for p in pairs:
    s = jaccard(char3[p[0]], char3[p[1]])
    print(p, lsh_prob(s,r,b))


LSH PROBABILITIES
('D1', 'D2') 0.9997216842914514
('D1', 'D3') 0.0001503031583247605
('D1', 'D4') 3.9039083077341274e-10
('D2', 'D3') 9.789532539605794e-05
('D2', 'D4') 4.118847485301558e-10
('D3', 'D4') 6.160671972565979e-10
